In [1]:
import boto3
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


inspection_id = "0A49KLT3B7Y" #fhr8
angle_profiles = ["0"]
num_tracks = 22
run_number = 8
results_dir = f"./data-files/insp_{inspection_id}_dent_detection_us_lora_fhr8"

In [2]:
def download_s3_folder(bucket_name: str, s3_folder: str, local_dir: str = None):
    """
    Download the contents of a folder directory
    Args:
        bucket_name: the name of the s3 bucket
        s3_folder: the folder path in the s3 bucket
        local_dir: the dir data is downloaded to
    """
    s3 = boto3.resource("s3")
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=s3_folder):
        target = (
            obj.key
            if local_dir is None
            else os.path.join(local_dir, os.path.relpath(obj.key, s3_folder))
        )
        if not os.path.exists(os.path.dirname(target)):
            os.makedirs(os.path.dirname(target))
        if obj.key[-1] == "/":
            continue
        if not os.path.exists(target):
            bucket.download_file(obj.key, target)
            print("Downloading: {}".format(obj.key))

if not os.path.exists(results_dir):
    os.makedirs(results_dir)

download_s3_folder("dv-test-prefect", f"track_runs/{inspection_id}/corrosion_detection/", results_dir)

Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000000000_000000050000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000050000_000000100000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000100000_000000150000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000150000_000000200000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000200000_000000250000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000250000_000000300000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/frames_000000300000_000000350000_000001.parquet
Downloading: track_runs/0A49KLT3B7Y/corrosion_detection/track_00/01-000-11YWD5E5/0/

KeyboardInterrupt: 

In [3]:

root = Path(results_dir)
data_files = sorted(root.rglob("*.parquet"))

In [4]:
len(data_files)

5211

In [5]:
data_files[-2]
df = pd.read_parquet(data_files[-2])
#remove attn_map columns
df = df.drop(columns=[col for col in df.columns if "attn_map" in col])
df.head()
#13488222

,inspection_id,clip_id,track_id,frame_idx,timer_tick,odometer_tick,lam_prob
0,0A49KLT3B7Y,01-021-11YWDR6A,21,17250000,485546212,261975096,0.006338
1,0A49KLT3B7Y,01-021-11YWDR6A,21,17250001,485546233,261975111,0.006338
2,0A49KLT3B7Y,01-021-11YWDR6A,21,17250002,485546255,261975126,0.006338
3,0A49KLT3B7Y,01-021-11YWDR6A,21,17250003,485546276,261975141,0.006338
4,0A49KLT3B7Y,01-021-11YWDR6A,21,17250004,485546299,261975156,0.006338


In [6]:
def keep_confident(df, threshold=0.5, score_col='lam_prob'):
    """
    Keep only rows where crack_score is greater than the given threshold.
    """
    return df[df[score_col] > threshold].reset_index(drop=True)

## Create a DataFrame for each profile and track and range
def create_df(track_idx, threshold = 0.7, frame_range=None):
    if frame_range is None:
        data_files_range = [f for f in data_files if f.parent.parent.parent.name == f"track_{track_idx:02d}"]
    else:
        data_files_range = [f for f in data_files if f.parent.parent.parent.name == f"track_{track_idx:02d}" and f.name.startswith(f"frames_{frame_range[0]:012d}_{frame_range[1]:012d}")]


    dfs = []
    for path in data_files_range:
        df = pd.read_parquet(path)
        df = keep_confident(df, threshold=threshold, score_col='lam_prob')
        df = df.drop(columns=[col for col in df.columns if "attn_map" in col])
        dfs.append(df)

        # Handle case where all DataFrames were empty or had errors
    if not dfs:
        print(f"No valid data found for frame_range={frame_range}")
        return pd.DataFrame()
    
    return pd.concat(dfs) if len(dfs) > 1 else dfs[0]

In [7]:
def analyze_frame_sequences(df, frame_col='frame_idx', range_col='frame_range', plot=True):
    """
    Analyze consecutive frame sequences in a DataFrame.
    
    Args:
        df: DataFrame with frame indices
        frame_col: column name containing frame indices
        range_col: name for the new range column
        plot: whether to show histogram of sequence lengths
    
    Returns:
        tuple: (df_with_ranges, num_unique_sequences, sequence_lengths)
    """
    if len(df) == 0:
        print("Empty DataFrame")
        return df.copy(), 0, []
    
    # # Sort by frame index and reset index
    # df_sorted = df.copy().sort_values(frame_col).reset_index(drop=True) #aleardy sorted
    df_sorted = df.copy()
    
    
    # Find breaks in consecutive sequences
    frame_diff = df_sorted[frame_col].diff()
    breaks = (frame_diff != 1) & (frame_diff.notna())
    
    # Assign range IDs - each break starts a new range
    range_ids = breaks.cumsum()
    
    # Create range labels and calculate sequence lengths
    range_labels = []
    sequence_lengths = []
    
    for range_id in range_ids.unique():
        mask = range_ids == range_id
        start_frame = df_sorted.loc[mask, frame_col].min()
        end_frame = df_sorted.loc[mask, frame_col].max()
        sequence_length = end_frame - start_frame + 1
        
        range_labels.append(f"{start_frame}_{end_frame}")
        sequence_lengths.append(sequence_length)
    
    # Map range IDs to labels
    range_mapping = dict(zip(range_ids.unique(), range_labels))
    df_sorted[range_col] = range_ids.map(range_mapping)
    
    # Number of unique sequences
    num_unique_sequences = len(range_labels)
    
    # Print summary
    print(f"Total frames: {len(df_sorted)}")
    print(f"Number of unique sequences: {num_unique_sequences}")
    print(f"Sequence lengths - Min: {min(sequence_lengths)}, Max: {max(sequence_lengths)}, Mean: {np.mean(sequence_lengths):.1f}")
    
    # Plot histogram if requested
    if plot:
        plt.figure(figsize=(10, 6))
        plt.hist(sequence_lengths, bins=max(1, min(50, num_unique_sequences//2)), 
                 color='skyblue', alpha=0.7, edgecolor='black')
        plt.title('Histogram of Consecutive Frame Sequence Lengths')
        plt.xlabel('Sequence Length (number of consecutive frames)')
        plt.ylabel('Frequency')
        plt.grid(True, alpha=0.3)
        
        # Add statistics text
        plt.text(0.7, 0.9, f'Total sequences: {num_unique_sequences}\n'
                           f'Mean length: {np.mean(sequence_lengths):.1f}\n'
                           f'Max length: {max(sequence_lengths)}', 
                 transform=plt.gca().transAxes, 
                 bbox=dict(boxstyle="round", facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()
    
    return df_sorted, num_unique_sequences, sequence_lengths

In [8]:
def merge_sequences_to_single_row(df_with_ranges, range_col='frame_range', inspection_id=None):
    """
    Merge rows by sequence to have one row per frame_range sequence.
    
    Args:
        df_with_ranges: DataFrame with frame_range column and detection data
        range_col: column name containing frame range identifiers
        inspection_id: inspection ID to add to the output (if not in df)
    
    Returns:
        DataFrame with one row per sequence containing aggregated information
    """
    if len(df_with_ranges) == 0:
        return pd.DataFrame()
    
    # Group by frame_range and aggregate
    agg_dict = {
        'frame_idx': ['min', 'max', 'count'],
        'lam_prob': ['mean', 'max', 'min', 'std'],
        'timer_tick': ['min', 'max'],
        'odometer_tick': ['min', 'max'],
        'clip_id': 'first',
        'track_id': 'first'
    }
    
    # Add inspection_id if it exists in the dataframe
    if 'inspection_id' in df_with_ranges.columns:
        agg_dict['inspection_id'] = 'first'
    
    sequence_summary = df_with_ranges.groupby(range_col).agg(agg_dict).round(4)
    
    # Flatten column names
    sequence_summary.columns = [f'{col[0]}_{col[1]}' if col[1] != '' else col[0] 
                               for col in sequence_summary.columns]
    
    # Reset index to make frame_range a regular column
    sequence_summary = sequence_summary.reset_index()
    
    # Add computed columns
    sequence_summary['sequence_length'] = sequence_summary['frame_idx_count']
    sequence_summary['frame_span'] = (sequence_summary['frame_idx_max'] - 
                                     sequence_summary['frame_idx_min'] + 1)
    sequence_summary['timer_span'] = (sequence_summary['timer_tick_max'] - 
                                     sequence_summary['timer_tick_min'])
    sequence_summary['odometer_span'] = (sequence_summary['odometer_tick_max'] - 
                                        sequence_summary['odometer_tick_min'])
    
    # Rename columns for clarity
    column_rename = {
        'frame_idx_min': 'start_frame',
        'frame_idx_max': 'end_frame', 
        'frame_idx_count': 'num_detections',
        'lam_prob_mean': 'avg_confidence',
        'lam_prob_max': 'max_confidence',
        'lam_prob_min': 'min_confidence',
        'lam_prob_std': 'confidence_std',
        'timer_tick_min': 'start_timer_tick',
        'timer_tick_max': 'end_timer_tick',
        'odometer_tick_min': 'start_odometer_tick',
        'odometer_tick_max': 'end_odometer_tick',
        'clip_id_first': 'clip_id',
        'track_id_first': 'track_id'
    }
    
    # Add inspection_id rename if it exists
    if 'inspection_id_first' in sequence_summary.columns:
        column_rename['inspection_id_first'] = 'inspection_id'
    
    sequence_summary = sequence_summary.rename(columns=column_rename)
    
    # Add inspection_id if provided and not in dataframe
    if inspection_id and 'inspection_id' not in sequence_summary.columns:
        sequence_summary['inspection_id'] = inspection_id
    
    # Reorder columns for better readability
    column_order = [
        'inspection_id', 'frame_range', 'clip_id', 'track_id',
        'start_frame', 'end_frame', 'sequence_length', 'frame_span', 'num_detections',
        'start_timer_tick', 'end_timer_tick', 'timer_span',
        'start_odometer_tick', 'end_odometer_tick', 'odometer_span',
        'avg_confidence', 'max_confidence', 'min_confidence', 'confidence_std'
    ]
    
    # Only keep columns that exist
    available_columns = [col for col in column_order if col in sequence_summary.columns]
    sequence_summary = sequence_summary[available_columns]
    
    print(f"Merged {len(df_with_ranges)} rows into {len(sequence_summary)} sequences")
    
    return sequence_summary

In [9]:
from io import BytesIO

def save_df_to_s3_parquet(df, bucket_name, s3_key):
    """
    Save DataFrame to S3 as parquet file
    
    Args:
        df: DataFrame to save
        bucket_name: S3 bucket name
        s3_key: S3 key (path) for the file
    """
    # Convert DataFrame to parquet in memory
    parquet_buffer = BytesIO()
    df.to_parquet(parquet_buffer, index=False)
    parquet_buffer.seek(0)
    
    # Upload to S3
    s3_client = boto3.client('s3')
    s3_client.put_object(
        Bucket=bucket_name,
        Key=s3_key,
        Body=parquet_buffer.getvalue(),
        ContentType='application/octet-stream'
    )
    
    print(f"DataFrame saved to s3://{bucket_name}/{s3_key}")

In [ ]:
threshold = 0.99
num_tracks = 22
bucket_name = "dent-dev"

for track_idx in range(0, 1):
    s3_key = f"dent_bookmarks/UltrasoundDent_bookmarks_{inspection_id}_track_{track_idx}.parquet"
    filtered_df = create_df(track_idx=track_idx, threshold=threshold)
    if filtered_df.empty:
        print(f"No data for track {track_idx} at threshold {threshold}, skipping.")
        continue
    # Create a histogram of lam_prob
    plt.style.use('dark_background')
    plt.figure(figsize=(10, 6))
    plt.hist(filtered_df['lam_prob'], bins=50, color='blue', alpha=0.7)
    plt.title('Histogram of Dent_prob')
    plt.xlabel('Dent_prob')
    plt.ylabel('Frequency')
    plt.grid(True)  

    count = len(filtered_df)
    df_with_ranges, num_sequences, seq_lengths = analyze_frame_sequences(filtered_df)


    sequence_df = merge_sequences_to_single_row(df_with_ranges, inspection_id=inspection_id)

    # Optional: Show summary statistics
    print(f"\nSequence Summary for track {track_idx} and threshold {threshold}:")
    print(f"Total sequences: {len(sequence_df)}")
    print(f"Average sequence length: {sequence_df['sequence_length'].mean():.1f}")
    print(f"Longest sequence: {sequence_df['sequence_length'].max()} frames")
    print(f"Average confidence: {sequence_df['avg_confidence'].mean():.3f}")
    display(sequence_df)

    #save_df_to_s3_parquet(sequence_df, bucket_name, s3_key)
